# ⚾ 레퍼토리 반사실 비교: 재대결에서 "뭘 던졌어야 했는가"

* **작성자:** DY
* **작성일:** 2026-10-05
* **목적:** 노트북 3(`03_DY_run_value_gbm_pitcher_tendency.ipynb`)의 메인 결론은 "회피 성향 자체가 장기적으로
  좋은 선택이었는가"였고, "이 한 타석에 뭘 던졌어야 했나"는 회의록 지침("일반 구종 추천 모델로 번지지 않게")에
  따라 메인 결론에서 의도적으로 뺐습니다(노트북 3의 10절에 보조 지표로만 남겨둠). 이 노트북은 그 질문을
  **독립적인 검증**으로 따로 다룹니다 — 두 질문은 서로 다릅니다:
  * 노트북 3: "회피하는 투수들이, 여러 투수·여러 타석에 걸쳐 평균적으로 더 나은 결과를 내는가" (성향 수준)
  * 이 노트북: "그 재대결 순간, 그 투수의 레퍼토리 중에서 실제로 던진 구종이 모델이 보기에 최선이었는가"
    (개별 결정 수준)
  같은 GBM(상황+구종+구사율, 노트북 3과 동일한 피처 — `pitcher_id`와 실행 품질은 여기서도 제외)을 쓰지만,
  평가 방식이 다릅니다: 재대결 결정구마다 그 투수·시즌 레퍼토리(구사율 5% 이상) 안의 각 구종을 똑같은 상황에
  대입해 예측하고, 실제 선택이 그중 최선과 얼마나 차이 나는지(gap) 봅니다.
* **설계 문서:** `docs/superpowers/specs/2026-10-03-run-value-gbm-pitcher-tendency-design.md` (노트북 3의
  6절과 공유; 이 노트북은 그 문서 10절을 메인 질문으로 승격한 별도 분석입니다)

In [1]:
# 1. 경로 설정 (src 폴더 접근용)
import sys, os
sys.path.append(os.path.abspath('..'))

# 2. Autoreload 설정
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from src.analysis.avoidance_stats import compute_season_usage_rate
from src.analysis.run_same_batter_rematch_analysis import build_xbh_rematch
from src.preprocessing.research_sample import KEY_COLUMNS, list_research_csvs

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)

MIN_USAGE_RATE = 0.05
MIN_REMATCH_EVENTS = 10
MIN_SEASON_EVENTS = 5
TRAIN_SEASONS = (2021, 2022, 2023, 2024, 2025)
VALID_SEASON = 2026
RANDOM_STATE = 20261003

Error importing in API mode: ImportError("dlopen(/Users/dongyunkwak/GitHub/9th-first-project-baseball-2/.venv/lib/python3.14/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): Library not loaded: /Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib\n  Referenced from: <9F0E20C3-D782-31B5-943E-66F62BCE77E8> /Users/dongyunkwak/GitHub/9th-first-project-baseball-2/.venv/lib/python3.14/site-packages/_rinterface_cffi_api.abi3.so\n  Reason: tried: '/Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib' (no such file), '/Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib' (no such file)")


Trying to import in ABI mode.


## 1. 팀 CSV 읽기

`data/raw` 없이 팀 research CSV 6개를 읽습니다. `woba_value`/`woba_denom`은 그 타석을 끝낸 결정구에만 채워져 있고
(실측 확인됨), 이게 GBM의 학습·평가 단위(타석이 아니라 결정구 하나)를 정합니다.

In [2]:
LOAD_COLUMNS = [
    'game_pk', 'game_date', 'at_bat_number', 'pitch_number', 'pitcher', 'player_name', 'batter',
    'stand', 'p_throws', 'pitch_type', 'events', 'balls', 'strikes', 'outs_when_up',
    'inning', 'inning_topbot', 'on_1b', 'on_2b', 'on_3b', 'home_score', 'away_score',
    'woba_value', 'woba_denom',
]


def load_team_csv() -> pd.DataFrame:
    frames = [
        pd.read_csv(path, usecols=LOAD_COLUMNS, encoding='utf-8-sig', low_memory=False)
        for path in list_research_csvs()
    ]
    pitches = pd.concat(frames, ignore_index=True)
    pitches['season'] = pd.to_datetime(pitches['game_date']).dt.year
    return pitches


pitches = load_team_csv()
assert len(pitches) == 3_592_302 and pitches['pitcher'].nunique() == 1_067
assert not pitches.duplicated(KEY_COLUMNS).any()
print(f'{len(pitches):,}행, 투수 {pitches["pitcher"].nunique():,}명')

3,592,302행, 투수 1,067명


## 2. 상황 피처

`score_diff`는 `src/preprocessing/build_next_ab_dataset.py`의 `compute_score_diff()`와 같은 정의(투수팀 관점,
수비팀 − 공격팀)를 벡터화한 것입니다.

In [3]:
SITUATION_COLUMNS = ['runners_n', 'risp', 'score_diff', 'platoon_match']


def add_situation_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    on1, on2, on3 = out['on_1b'].notna(), out['on_2b'].notna(), out['on_3b'].notna()
    out['runners_n'] = on1.astype(int) + on2.astype(int) + on3.astype(int)
    out['risp'] = (on2 | on3).astype(int)
    out['score_diff'] = np.where(
        out['inning_topbot'] == 'Top', out['home_score'] - out['away_score'], out['away_score'] - out['home_score']
    )
    out['platoon_match'] = np.where(
        out['stand'].isna() | out['p_throws'].isna(), np.nan, (out['stand'] == out['p_throws']).astype(float)
    )
    return out

In [4]:
_t = pd.DataFrame({
    'on_1b': [np.nan, 1.0, np.nan], 'on_2b': [np.nan, np.nan, 2.0], 'on_3b': [np.nan, np.nan, np.nan],
    'inning_topbot': ['Top', 'Bot', 'Top'], 'home_score': [3, 5, 2], 'away_score': [3, 2, 4],
    'stand': ['L', 'R', 'R'], 'p_throws': ['R', 'R', 'L'],
})
_r = add_situation_columns(_t)
assert _r['runners_n'].tolist() == [0, 1, 1]
assert _r['risp'].tolist() == [0, 0, 1]
assert _r['score_diff'].tolist() == [0, -3, -2]
assert _r['platoon_match'].tolist() == [0.0, 1.0, 0.0]
print('add_situation_columns 테스트 통과')

add_situation_columns 테스트 통과


In [5]:
pitches = add_situation_columns(pitches)
print(pitches[SITUATION_COLUMNS].describe().round(3))

         runners_n         risp   score_diff  platoon_match
count  3592302.000  3592302.000  3592302.000    3592302.000
mean         0.580        0.237        0.189          0.447
std          0.775        0.425        2.936          0.497
min          0.000        0.000      -25.000          0.000
25%          0.000        0.000       -1.000          0.000
50%          0.000        0.000        0.000          0.000
75%          1.000        0.000        2.000          1.000
max          3.000        1.000       24.000          1.000


## 3. 재대결 이벤트와 투수별 회피율

장타 재대결 이벤트는 노트북 01·02와 같은 건수(27,391건)가 나와야 합니다. 회피율 = 1 − 평균(`reused_same_type`).

In [6]:
season_usage = compute_season_usage_rate(pitches)
xbh = build_xbh_rematch(pitches, season_usage)
xbh = xbh[xbh['has_next_ab']].copy()
assert len(xbh) == 27_391, len(xbh)
print(f'재대결 이벤트 {len(xbh):,}건')

2026-10-05 15:25:38,697 [INFO] 장타 70319건 중 같은 타자·같은 투수 재대결 27391건 (39.0%)
        xbh_total  same_batter_rematch  rematch_share
season                                               
2021        12709                 4688         0.3689
2022        12186                 4623         0.3794
2023        13137                 5280         0.4019
2024        12427                 4977         0.4005
2025        12394                 4916         0.3966
2026         7466                 2907         0.3894


재대결 이벤트 27,391건


In [7]:
def _agg_rate(events: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    g = events.groupby(group_cols)['reused_same_type']
    out = g.agg(n_events='size', reuse_rate='mean').reset_index()
    out['avoidance_rate'] = 1 - out['reuse_rate']
    return out.drop(columns='reuse_rate')


def season_rate(events: pd.DataFrame) -> pd.DataFrame:
    return _agg_rate(events, ['pitcher', 'season'])


def pitcher_rate(events: pd.DataFrame) -> pd.DataFrame:
    return _agg_rate(events, ['pitcher'])

In [8]:
_e = pd.DataFrame({
    'pitcher': [1, 1, 1, 2, 2], 'season': [2021, 2021, 2022, 2021, 2021],
    'reused_same_type': [1, 0, 0, 1, 1],
})
_s = season_rate(_e)
assert _s.set_index(['pitcher', 'season'])['n_events'].to_dict() == {(1, 2021): 2, (1, 2022): 1, (2, 2021): 2}
assert np.isclose(_s.set_index(['pitcher', 'season']).loc[(1, 2021), 'avoidance_rate'], 0.5)
_p = pitcher_rate(_e)
assert _p.set_index('pitcher')['n_events'].to_dict() == {1: 3, 2: 2}
assert np.isclose(_p.set_index('pitcher').loc[1, 'avoidance_rate'], 1 - 1 / 3)
assert np.isclose(_p.set_index('pitcher').loc[2, 'avoidance_rate'], 0.0)
print('회피율 집계 테스트 통과')

회피율 집계 테스트 통과


## 4. 포함 기준과 그룹화

노트북 3과 동일하게 투수 1명(2021–2026 전체 합산) 단위, 재대결 10건 이상, 3분위(하위=승부형/상위=회피형/
중위 제외)로 그룹화합니다. 이 그룹은 이 노트북에서 메인 지표가 아니라 — 아래 레퍼토리 반사실 결과를
"회피형과 승부형이 개별 결정 수준에서도 다른가"로 한 번 더 나눠보는 보조적 렌즈로만 씁니다. 시즌 안정성
검증(ω²=0.266)과 그 함의는 노트북 3의 해석을 참고하세요 — 여기서는 되풀이하지 않습니다.

In [9]:
def assign_tercile(rates: pd.DataFrame) -> pd.DataFrame:
    out = rates.copy()
    q1, q2 = out['avoidance_rate'].quantile([1 / 3, 2 / 3])
    out['tercile_group'] = np.select(
        [out['avoidance_rate'] <= q1, out['avoidance_rate'] >= q2], ['승부형', '회피형'], default='중립'
    )
    return out


def assign_median(rates: pd.DataFrame) -> pd.DataFrame:
    out = rates.copy()
    med = out['avoidance_rate'].median()
    out['median_group'] = np.where(out['avoidance_rate'] >= med, '회피형', '승부형')
    return out

In [10]:
_r = pd.DataFrame({'pitcher': range(6), 'avoidance_rate': [0.1, 0.2, 0.3, 0.7, 0.8, 0.9]})
_t = assign_tercile(_r)
assert _t.set_index('pitcher')['tercile_group'].to_dict() == {
    0: '승부형', 1: '승부형', 2: '중립', 3: '중립', 4: '회피형', 5: '회피형',
}
_m = assign_median(_r)
assert set(_m.loc[_m['avoidance_rate'] <= 0.3, 'median_group']) == {'승부형'}
assert set(_m.loc[_m['avoidance_rate'] >= 0.7, 'median_group']) == {'회피형'}
print('그룹화 테스트 통과')

그룹화 테스트 통과


In [11]:
rates = pitcher_rate(xbh)
rates = rates[rates['n_events'] >= MIN_REMATCH_EVENTS].copy()
rates = assign_tercile(rates)
print(f'포함 기준(재대결 {MIN_REMATCH_EVENTS}건 이상) 만족 투수: {len(rates)}명')
print('3분위 그룹별 투수 수:', rates['tercile_group'].value_counts().to_dict())

포함 기준(재대결 10건 이상) 만족 투수: 439명
3분위 그룹별 투수 수: {'회피형': 147, '승부형': 147, '중립': 145}


## 5. GBM 학습 테이블

표본 전체(재대결에 국한하지 않음)의 모든 결정구에서, 그 결정구의 구종이 그 투수·시즌에서 차지하는
`season_usage_rate`를 붙입니다. `pitcher_id`는 피처에 넣지 않습니다.

In [12]:
MODEL_FEATURES = [
    'pitch_type', 'stand', 'p_throws', 'balls', 'strikes', 'outs_when_up', 'runners_n', 'risp', 'score_diff',
    'season_usage_rate',
]
CATEGORICAL_FEATURES = ['pitch_type', 'stand', 'p_throws']

assert 'balls' in MODEL_FEATURES and 'strikes' in MODEL_FEATURES, (
    '스펙 4.2절: 볼카운트(balls/strikes)도 피처여야 함 -- 리뷰에서 누락이 지적됨'
)


def build_decisive_pitch_table(pitches_sit: pd.DataFrame, season_usage_: pd.DataFrame) -> pd.DataFrame:
    decisive = pitches_sit[pitches_sit['woba_value'].notna() & pitches_sit['pitch_type'].notna()].copy()
    decisive = decisive.merge(
        season_usage_[['pitcher', 'season', 'pitch_type', 'season_usage_rate']],
        on=['pitcher', 'season', 'pitch_type'], how='left',
    )
    for col in CATEGORICAL_FEATURES:
        decisive[col] = decisive[col].astype('category')
    return decisive

In [13]:
_px = pd.DataFrame({
    'pitcher': [1, 1, 1, 1], 'season': [2021, 2021, 2021, 2021], 'pitch_type': ['SL', 'FF', 'SL', None],
    'events': [None, None, 'single', 'walk'], 'woba_value': [np.nan, np.nan, 0.9, 0.7], 'woba_denom': [np.nan, np.nan, 1.0, 1.0],
    'outs_when_up': [0, 0, 0, 0], 'balls': [0, 0, 1, 4], 'strikes': [0, 0, 2, 0],
    'runners_n': [0, 0, 0, 0], 'risp': [0, 0, 0, 0], 'score_diff': [0, 0, 0, 0],
    'platoon_match': [1.0, 1.0, 1.0, 1.0], 'stand': ['R', 'R', 'R', 'R'], 'p_throws': ['R', 'R', 'R', 'R'],
})
_su = pd.DataFrame({'pitcher': [1, 1], 'season': [2021, 2021], 'pitch_type': ['SL', 'FF'], 'season_usage_rate': [0.6, 0.4]})
_tbl = build_decisive_pitch_table(_px, _su)
assert len(_tbl) == 1  # 결정구(woba_value 있고 pitch_type도 있는 행)만 1건 -- 4행은 woba_value는 있지만 pitch_type이 없어 제외
assert _tbl.iloc[0]['pitch_type'] == 'SL' and np.isclose(_tbl.iloc[0]['season_usage_rate'], 0.6)
assert _tbl.iloc[0]['balls'] == 1 and _tbl.iloc[0]['strikes'] == 2
assert str(_tbl['pitch_type'].dtype) == 'category' and str(_tbl['stand'].dtype) == 'category'
print('build_decisive_pitch_table 테스트 통과')

build_decisive_pitch_table 테스트 통과


In [14]:
decisive_table = build_decisive_pitch_table(pitches, season_usage)
assert decisive_table['season_usage_rate'].notna().mean() > 0.99, '구사율 매칭 실패 비율이 높음'
print(f'결정구 {len(decisive_table):,}건, season_usage_rate 결측 {decisive_table["season_usage_rate"].isna().sum()}건')

결정구 917,717건, season_usage_rate 결측 0건


## 6. GBM 학습과 검증

2021–2025로 학습, 2026으로 검증합니다. HGBR은 제곱오차를 최소화하도록 학습되므로 평가도 그에 맞는 R²/MSE를
기준으로 삼고, MAE는 참고로만 같이 봅니다(MAE만 보면 평균 쪽으로 쏠린 예측이 극단값 예측보다 유리해 보일 수
있어, 베이스라인보다 MAE가 낮다고 해서 모델이 분산을 실제로 설명한다는 뜻은 아닙니다 — 리뷰에서 지적된 점).

In [15]:
def fit_run_value_model(table: pd.DataFrame, seasons) -> HistGradientBoostingRegressor:
    train = table[table['season'].isin(seasons)]
    model = HistGradientBoostingRegressor(
        categorical_features='from_dtype', random_state=RANDOM_STATE, max_iter=200, early_stopping=True,
    )
    model.fit(train[MODEL_FEATURES], train['woba_value'])
    return model

In [16]:
train_set = decisive_table[decisive_table['season'].isin(TRAIN_SEASONS)]
valid_set = decisive_table[decisive_table['season'] == VALID_SEASON]
model_cv = fit_run_value_model(decisive_table, TRAIN_SEASONS)
pred_valid = model_cv.predict(valid_set[MODEL_FEATURES])
baseline_mae = mean_absolute_error(valid_set['woba_value'], np.full(len(valid_set), train_set['woba_value'].mean()))
model_mae = mean_absolute_error(valid_set['woba_value'], pred_valid)
model_mse = mean_squared_error(valid_set['woba_value'], pred_valid)
model_r2 = r2_score(valid_set['woba_value'], pred_valid)
print(f'베이스라인(평균 예측) MAE: {baseline_mae:.4f}')
print(f'GBM MAE: {model_mae:.4f} (베이스라인 대비 {(baseline_mae - model_mae) / baseline_mae:.1%}, 참고용)')
print(f'GBM MSE: {model_mse:.4f}, R²: {model_r2:.4f} (평균만 예측하면 R²=0)')
assert model_r2 > 0, 'GBM이 평균만 예측하는 것보다 분산을 못 줄임 -- 피처/학습을 재검토해야 함'

final_model = fit_run_value_model(decisive_table, decisive_table['season'].unique())
print('전체 데이터로 재학습한 최종 모델 준비 완료')

베이스라인(평균 예측) MAE: 0.4322
GBM MAE: 0.3989 (베이스라인 대비 7.7%, 참고용)
GBM MSE: 0.2445, R²: 0.0568 (평균만 예측하면 R²=0)


전체 데이터로 재학습한 최종 모델 준비 완료


## 7. 재대결 결정구 찾기

재대결 타석(이벤트의 `next_at_bat_number`)의 결정구를 찾습니다. `RVAE = 실제 woba_value − 모델 기대 Run
Value`(낮을수록 투수에게 유리)도 같이 구해 두는데, 이 노트북에서는 메인 지표가 아니라 10절 투수별 사례 표에
"그 결과가 기대보다 좋았는지"를 참고로 같이 보여주는 용도입니다 — 메인 질문은 "그 선택이 레퍼토리 중
최선이었는가"입니다.

In [17]:
def find_rematch_decisive_pitch(xbh_: pd.DataFrame, pitches_sit: pd.DataFrame) -> pd.DataFrame:
    decisive = pitches_sit[pitches_sit['woba_value'].notna()][
        ['game_pk', 'pitcher', 'season', 'at_bat_number', *MODEL_FEATURES, 'woba_value']
    ]
    keys = xbh_[['game_pk', 'pitcher', 'next_at_bat_number']].rename(columns={'next_at_bat_number': 'at_bat_number'})
    out = keys.merge(decisive, on=['game_pk', 'pitcher', 'at_bat_number'], how='left', validate='many_to_one')
    return out


def compute_rvae(df: pd.DataFrame, model) -> pd.Series:
    expected = model.predict(df[MODEL_FEATURES])
    return df['woba_value'] - expected

In [18]:
_xbh_fix = pd.DataFrame({'game_pk': [100, 100], 'pitcher': [10, 10], 'next_at_bat_number': [5.0, 9.0]})
_pt_fix = pd.DataFrame({
    'game_pk': [100, 100, 100], 'pitcher': [10, 10, 10], 'season': [2021, 2021, 2021],
    'at_bat_number': [5, 5, 9], 'woba_value': [np.nan, 0.9, np.nan],
    'pitch_type': [None, 'SL', None], 'stand': ['R', 'R', 'R'], 'p_throws': ['R', 'R', 'R'],
    'balls': [0, 1, 0], 'strikes': [0, 2, 0], 'outs_when_up': [0, 0, 0], 'runners_n': [0, 0, 0],
    'risp': [0, 0, 0], 'score_diff': [0, 0, 0], 'season_usage_rate': [np.nan, 0.5, np.nan],
})
_found = find_rematch_decisive_pitch(_xbh_fix, _pt_fix)
assert len(_found) == 2
assert _found.iloc[0]['season'] == 2021 and np.isclose(_found.iloc[0]['woba_value'], 0.9)  # season 컬럼이 살아있어야 함
assert pd.isna(_found.iloc[1]['woba_value'])  # at_bat 9엔 결정구가 없음 -- 못 찾은 경우가 NaN 행으로 남아야 함(조용히 빠지면 안 됨)
print('find_rematch_decisive_pitch 테스트 통과')

find_rematch_decisive_pitch 테스트 통과


In [19]:
class _StubModel:
    def predict(self, X):
        return np.full(len(X), 0.3)

_df = pd.DataFrame({'woba_value': [0.9, 0.0], **{c: [0] * 2 for c in MODEL_FEATURES}})
_rvae = compute_rvae(_df, _StubModel())
assert np.allclose(_rvae.to_numpy(), [0.6, -0.3])
print('compute_rvae 테스트 통과')

compute_rvae 테스트 통과


In [20]:
rematch_decisive = find_rematch_decisive_pitch(xbh, decisive_table)
missing = rematch_decisive['woba_value'].isna().sum()
print(f'재대결 {len(rematch_decisive):,}건 중 결정구를 못 찾은 건수: {missing} ({missing / len(rematch_decisive):.2%})')

# 왜 못 찾았는지 원인을 나눠 본다: (a) 그 타석 자체가 raw에 없음(시즌 마지막 날짜에서 끊긴 미완결 PA 등)
# (b) 타석은 있고 woba_value도 있지만 결정구의 pitch_type이 없어 decisive_table 단계에서 빠짐(예: 고의사구)
raw_decisive = pitches.loc[pitches['woba_value'].notna(), ['game_pk', 'pitcher', 'at_bat_number', 'pitch_type', 'events']]
missing_keys = rematch_decisive.loc[rematch_decisive['woba_value'].isna(), ['game_pk', 'pitcher', 'at_bat_number']]
missing_detail = missing_keys.merge(raw_decisive, on=['game_pk', 'pitcher', 'at_bat_number'], how='left')
missing_reason = np.select(
    [missing_detail['events'].isna(), missing_detail['pitch_type'].isna()],
    ['타석 자체가 없음(미완결 등)', '구종 미분류(고의사구 등)'], default='기타',
)
print('결정구 못 찾은 원인별 건수:', pd.Series(missing_reason).value_counts().to_dict())
no_type_events = missing_detail.loc[missing_detail['pitch_type'].isna() & missing_detail['events'].notna(), 'events']
if len(no_type_events):
    print('  구종 미분류의 이벤트 종류:', no_type_events.value_counts().to_dict())

rematch_decisive = rematch_decisive.dropna(subset=['woba_value'])
rematch_decisive['rvae'] = compute_rvae(rematch_decisive, final_model)

pitcher_rvae = rematch_decisive.groupby('pitcher')['rvae'].mean().rename('mean_rvae').reset_index()
rates = rates.merge(pitcher_rvae, on='pitcher', how='left')
assert rates['mean_rvae'].notna().all(), '포함 기준을 만족한 투수 중 RVAE가 없는 경우가 있음'

재대결 27,391건 중 결정구를 못 찾은 건수: 135 (0.49%)
결정구 못 찾은 원인별 건수: {'구종 미분류(고의사구 등)': 107, '타석 자체가 없음(미완결 등)': 28}
  구종 미분류의 이벤트 종류: {'intent_walk': 90, 'walk': 7, 'field_out': 4, 'strikeout': 4, 'field_error': 1, 'triple': 1}


## 8. 레퍼토리 반사실 비교 (메인 질문)

같은 상황에서 그 투수·시즌의 레퍼토리(구사율 5% 이상 **이고 학습 데이터에도 등장한** 구종) 안의 각 구종을
대입해 예측하고, 실제 선택이 레퍼토리 최선과 얼마나 차이 나는지(gap) 봅니다. 레퍼토리가 1개 구종뿐이면
비교 불가(`None`). **각 후보 구종은 자기 자신의 구사율로 평가합니다**(실제 투구의 구사율을 복사해 쓰지 않음 —
리뷰에서 지적된 버그).

In [21]:
def build_pitcher_repertoire(season_usage_: pd.DataFrame, min_rate: float, known_types: set) -> dict:
    """(pitcher, season) -> {pitch_type: season_usage_rate}. 구사율이 min_rate 이상이고
    known_types(학습 데이터가 실제로 본 구종)에도 있는 구종만 포함한다."""
    qualified = season_usage_[
        (season_usage_['season_usage_rate'] >= min_rate) & season_usage_['pitch_type'].isin(known_types)
    ]
    out = {}
    for key, group in qualified.groupby(['pitcher', 'season']):
        out[key] = dict(zip(group['pitch_type'], group['season_usage_rate']))
    return out


def repertoire_gap(row: pd.Series, repertoire: dict, model) -> float | None:
    type_rates = repertoire.get((row['pitcher'], row['season']), {})
    if len(type_rates) < 2:
        return None
    candidates = pd.DataFrame([row[MODEL_FEATURES].to_dict()] * len(type_rates))
    candidates['pitch_type'] = list(type_rates.keys())
    candidates['season_usage_rate'] = list(type_rates.values())  # 후보마다 자기 구사율 사용
    for col in CATEGORICAL_FEATURES:
        candidates[col] = candidates[col].astype('category')
    preds = model.predict(candidates[MODEL_FEATURES])
    actual_pred = model.predict(pd.DataFrame([row[MODEL_FEATURES].to_dict()]))[0]
    return float(actual_pred - preds.min())

In [22]:
_su2 = pd.DataFrame({
    'pitcher': [1, 1, 1], 'season': [2021, 2021, 2021], 'pitch_type': ['SL', 'FF', 'CU'],
    'season_usage_rate': [0.5, 0.45, 0.02],
})
_repertoire = build_pitcher_repertoire(_su2, MIN_USAGE_RATE, {'SL', 'FF', 'CU'})
assert _repertoire[(1, 2021)] == {'SL': 0.5, 'FF': 0.45}  # CU는 2% < 5% 기준으로 제외

_su3 = pd.DataFrame({'pitcher': [2, 2], 'season': [2021, 2021], 'pitch_type': ['SL', 'XX'], 'season_usage_rate': [0.5, 0.45]})
_repertoire_known = build_pitcher_repertoire(_su3, MIN_USAGE_RATE, {'SL'})  # XX는 구사율은 충분하지만 학습 데이터엔 없음
assert list(_repertoire_known[(2, 2021)].keys()) == ['SL']  # XX 제외 -> 레퍼토리 1개

_row = pd.Series({'pitcher': 2, 'season': 2021, 'pitch_type': 'SL', 'stand': 'R', 'p_throws': 'R',
                   'balls': 0, 'strikes': 0, 'outs_when_up': 0, 'runners_n': 0, 'risp': 0, 'score_diff': 0,
                   'season_usage_rate': 0.5})
assert repertoire_gap(_row, _repertoire_known, None) is None  # 레퍼토리 1개 -> 비교 불가

_su4 = pd.DataFrame({'pitcher': [1, 1], 'season': [2021, 2021], 'pitch_type': ['SL', 'FF'], 'season_usage_rate': [0.5, 0.1]})
_repertoire2 = build_pitcher_repertoire(_su4, MIN_USAGE_RATE, {'SL', 'FF'})

class _StubModel2:
    def predict(self, X):
        return X['season_usage_rate'].to_numpy()  # 구사율을 그대로 예측값으로 반환 -> 후보별 구사율이 안 쓰이면 들통남

_row1 = pd.Series({'pitcher': 1, 'season': 2021, 'pitch_type': 'SL', 'stand': 'R', 'p_throws': 'R',
                    'balls': 0, 'strikes': 0, 'outs_when_up': 0, 'runners_n': 0, 'risp': 0, 'score_diff': 0,
                    'season_usage_rate': 0.5})
_gap = repertoire_gap(_row1, _repertoire2, _StubModel2())
assert np.isclose(_gap, 0.4), _gap  # 실제(SL, 구사율0.5->예측0.5) - 레퍼토리최선(FF, 구사율0.1->예측0.1) = 0.4
# (버그가 남아 있었다면 모든 후보가 실제 투구의 구사율 0.5를 복사해 받아 예측이 전부 0.5가 되고, gap=0이 나왔을 것)
print('repertoire_gap 테스트 통과')

repertoire_gap 테스트 통과


In [23]:
def batch_repertoire_gap(events: pd.DataFrame, repertoire: dict, model, actual_preds: np.ndarray) -> pd.Series:
    """repertoire_gap과 수학적으로 같은 결과를 내지만, 이벤트마다 model.predict()를 따로 부르지 않고
    모든 후보를 한 번에 모아 단 한 번(또는 소수)만 호출한다 -- 행 단위 .apply()가 모델 호출 수만 번의
    스레드 오버헤드로 병목이 됐기 때문(리뷰 이후 발견). `actual_preds`는 events와 같은 길이로, events의
    실제 선택에 대한 예측값(이미 compute_rvae에서 계산한 것을 재사용)."""
    events_idx = events.reset_index(drop=True).reset_index(names='_event_idx')
    repertoire_long = pd.DataFrame(
        [(p, s, pt, rate) for (p, s), types in repertoire.items() for pt, rate in types.items()],
        columns=['pitcher', 'season', 'pitch_type', 'season_usage_rate'],
    )
    repertoire_size = repertoire_long.groupby(['pitcher', 'season']).size().rename('n_types')
    eligible_keys = repertoire_size[repertoire_size >= 2].index  # 레퍼토리 1개뿐이면 비교 불가(None과 동일)

    other_cols = [c for c in MODEL_FEATURES if c not in ('pitch_type', 'season_usage_rate')]
    candidates = events_idx[['_event_idx', 'pitcher', 'season', *other_cols]].merge(
        repertoire_long, on=['pitcher', 'season'], how='inner'
    )
    candidates = candidates[candidates.set_index(['pitcher', 'season']).index.isin(eligible_keys)]
    if candidates.empty:
        return pd.Series(np.nan, index=events.index)

    for col in CATEGORICAL_FEATURES:
        candidates[col] = candidates[col].astype(events[col].dtype)  # 학습 때와 같은 카테고리 보장
    candidates['_pred'] = model.predict(candidates[MODEL_FEATURES])
    best = candidates.groupby('_event_idx')['_pred'].min()

    gap = pd.Series(np.nan, index=events_idx['_event_idx'])
    gap.loc[best.index] = actual_preds[best.index.to_numpy()] - best.to_numpy()
    gap.index = events.index
    return gap


_events_small = pd.DataFrame({
    'pitcher': [1, 1, 2], 'season': [2021, 2021, 2021], 'pitch_type': ['SL', 'SL', 'SL'],
    'stand': ['R', 'R', 'R'], 'p_throws': ['R', 'R', 'R'], 'balls': [0, 0, 0], 'strikes': [0, 0, 0],
    'outs_when_up': [0, 0, 0], 'runners_n': [0, 0, 0], 'risp': [0, 0, 0], 'score_diff': [0, 0, 0],
    'season_usage_rate': [0.5, 0.5, 0.5],
})
_repertoire_small = {(1, 2021): {'SL': 0.5, 'FF': 0.1}}  # pitcher 2는 레퍼토리에 아예 없음 -> 비교 불가

class _StubModel4:
    def predict(self, X):
        return X['season_usage_rate'].to_numpy()

_stub4 = _StubModel4()
_actual_preds_small = _stub4.predict(_events_small[MODEL_FEATURES])
_batched = batch_repertoire_gap(_events_small, _repertoire_small, _stub4, _actual_preds_small)
_row_wise = _events_small.apply(lambda r: repertoire_gap(r, _repertoire_small, _stub4), axis=1)
assert np.allclose(_batched.fillna(-999), _row_wise.fillna(-999)), (_batched.tolist(), _row_wise.tolist())
print('batch_repertoire_gap 테스트 통과 (행 단위 기준과 일치)')

batch_repertoire_gap 테스트 통과 (행 단위 기준과 일치)


In [24]:
known_pitch_types = set(decisive_table['pitch_type'].cat.categories)
dropped_by_known = season_usage[
    (season_usage['season_usage_rate'] >= MIN_USAGE_RATE) & ~season_usage['pitch_type'].isin(known_pitch_types)
]
print(f'구사율 기준은 넘지만 학습 데이터에 없어 레퍼토리에서 빠진 투수-시즌-구종: {len(dropped_by_known)}건')

pitcher_repertoire = build_pitcher_repertoire(season_usage, MIN_USAGE_RATE, known_pitch_types)
# repertoire_gap(행 단위)은 위 테스트로 정확성이 검증된 기준 구현이다. 실제 27,391건에는 model.predict()를
# 행마다(사실상 건당 2번) 따로 부르면 수만 번의 작은 호출이 스레드 풀 오버헤드로 치명적인 병목이 되어
# (18분 넘게 안 끝남을 실측 확인) batch_repertoire_gap(한두 번의 배치 predict)으로 대체한다.
actual_preds_rematch = final_model.predict(rematch_decisive[MODEL_FEATURES])
gaps = batch_repertoire_gap(rematch_decisive, pitcher_repertoire, final_model, actual_preds_rematch)
rematch_decisive['repertoire_gap'] = gaps
not_comparable = gaps.isna().sum()
print(f'레퍼토리 비교 불가(구종 1개 이하) {not_comparable}건 / 전체 {len(gaps)}건')

comparable = rematch_decisive.dropna(subset=['repertoire_gap']).copy()
comparable['matched_best'] = np.isclose(comparable['repertoire_gap'], 0.0)

# 참고용: 레퍼토리 크기가 n이면 무작위로 찍어도 최선과 일치할 확률은 평균적으로 1/n
repertoire_size_table = pd.DataFrame(
    [(p, s, len(types)) for (p, s), types in pitcher_repertoire.items()],
    columns=['pitcher', 'season', 'repertoire_size'],
)
repertoire_sizes = rematch_decisive.merge(repertoire_size_table, on=['pitcher', 'season'], how='left')['repertoire_size'].fillna(0)
chance_match_rate = (1 / repertoire_sizes[repertoire_sizes >= 2]).mean()
print(f'레퍼토리 평균 크기 {repertoire_sizes[repertoire_sizes >= 2].mean():.2f}개, 무작위 선택 시 기대 일치율 {chance_match_rate:.4f}')

pitcher_gap = rematch_decisive.groupby('pitcher')['repertoire_gap'].mean().rename('mean_repertoire_gap').reset_index()
match_rate = comparable.groupby('pitcher')['matched_best'].mean().rename('best_match_rate').reset_index()
pitcher_gap = pitcher_gap.merge(match_rate, on='pitcher', how='left')
rates = rates.merge(pitcher_gap, on='pitcher', how='left')
display(rates[['pitcher', 'n_events', 'avoidance_rate', 'mean_rvae', 'mean_repertoire_gap', 'best_match_rate']].describe().round(4))

print('전체 재대결 결정구의 레퍼토리 gap 분포 (0 = 레퍼토리 중 최선을 선택):')
display(comparable['repertoire_gap'].describe().round(4))

구사율 기준은 넘지만 학습 데이터에 없어 레퍼토리에서 빠진 투수-시즌-구종: 0건


레퍼토리 비교 불가(구종 1개 이하) 0건 / 전체 27256건
레퍼토리 평균 크기 4.44개, 무작위 선택 시 기대 일치율 0.2366


,pitcher,n_events,avoidance_rate,mean_rvae,mean_repertoire_gap,best_match_rate
count,439.0000,439.0000,439.0000,439.0000,439.0000,439.0000
mean,640213.5194,60.3212,0.5365,0.0265,0.0155,0.3339
std,63943.2351,47.4195,0.1300,0.1028,0.0043,0.1045
min,425794.0000,10.0000,0.1000,-0.2743,0.0032,0.0000
25%,607634.5000,21.0000,0.4454,-0.0316,0.0126,0.2663
50%,663623.0000,45.0000,0.5500,0.0207,0.0154,0.3282
75%,676976.5000,87.5000,0.6205,0.0808,0.0180,0.3921
max,813349.0000,204.0000,0.9231,0.6390,0.0340,0.8571


전체 재대결 결정구의 레퍼토리 gap 분포 (0 = 레퍼토리 중 최선을 선택):


count    27256.0000
mean         0.0157
std          0.0192
min         -0.0261
25%          0.0000
50%          0.0098
75%          0.0254
max          0.3122
Name: repertoire_gap, dtype: float64

## 9. 개별 타석 사례: 구체적으로 뭘 던졌어야 했는가

비교 가능한 재대결 결정구 중, 모델 기준으로 **실제 선택이 레퍼토리 최선과 가장 크게 벌어진** 5건입니다.
"그 상황에서 그 투수의 레퍼토리 중 무엇이 최선이었는가"를 실제 사례로 보여줍니다 — 이게 이 노트북이 메인
결론으로 쓰지 않는(회의록 지침) "개별 타석 최적 구종"을, 그래도 구체적으로 보고 싶을 때 보는 자리입니다.

In [25]:
def repertoire_candidates_detail(row: pd.Series, repertoire: dict, model) -> pd.DataFrame:
    """repertoire_gap과 같은 입력으로, 후보 구종마다 예측값을 보여주는 표를 만든다 (사례 설명용)."""
    type_rates = repertoire.get((row['pitcher'], row['season']), {})
    candidates = pd.DataFrame([row[MODEL_FEATURES].to_dict()] * len(type_rates))
    candidates['pitch_type'] = list(type_rates.keys())
    candidates['season_usage_rate'] = list(type_rates.values())
    for col in CATEGORICAL_FEATURES:
        candidates[col] = candidates[col].astype('category')
    candidates['predicted_run_value'] = model.predict(candidates[MODEL_FEATURES])
    return candidates[['pitch_type', 'season_usage_rate', 'predicted_run_value']].sort_values('predicted_run_value')


worst_cases = comparable.sort_values('repertoire_gap', ascending=False).head(5)
for i, (_, case) in enumerate(worst_cases.iterrows(), start=1):
    name = xbh.loc[xbh['pitcher'] == case['pitcher'], 'pitcher_name'].iloc[0]
    print(
        f"\n사례 {i}: {name} ({int(case['season'])}년) -- "
        f"{int(case['balls'])}B{int(case['strikes'])}S, 아웃 {int(case['outs_when_up'])}, "
        f"주자 {int(case['runners_n'])}명, 득점차 {int(case['score_diff']):+d}, "
        f"상대 {'동타' if case['stand'] == case['p_throws'] else '반대타'}"
    )
    detail = repertoire_candidates_detail(case, pitcher_repertoire, final_model)
    actual_type = case['pitch_type']
    detail = detail.assign(실제선택=np.where(detail['pitch_type'] == actual_type, '←실제', ''))
    if not (detail['pitch_type'] == actual_type).any():
        # 실제 던진 구종이 그 시즌 구사율 5% 미만이라 "레퍼토리"엔 안 들어감 -- 그래도 실제 선택이 뭐였는지는 보여준다
        actual_pred = final_model.predict(pd.DataFrame([case[MODEL_FEATURES].to_dict()]))[0]
        actual_row = pd.DataFrame([{
            'pitch_type': actual_type, 'season_usage_rate': case['season_usage_rate'],
            'predicted_run_value': actual_pred, '실제선택': '←실제(구사율 5% 미만이라 레퍼토리 밖)',
        }])
        detail = pd.concat([detail, actual_row], ignore_index=True).sort_values('predicted_run_value')
    print(f"  실제 결과(woba_value): {case['woba_value']:.3f}, gap(실제 예측값 - 최선 예측값): {case['repertoire_gap']:.4f}")
    print(detail.round(4).to_string(index=False))


사례 1: Steele, Justin (2023년) -- 0B0S, 아웃 0, 주자 1명, 득점차 -1, 상대 반대타


  실제 결과(woba_value): 0.900, gap(실제 예측값 - 최선 예측값): 0.3122
pitch_type  season_usage_rate  predicted_run_value                    실제선택
        FF             0.6281               0.4031                        
        SL             0.3383               0.4277                        
        CU             0.0075               0.7153 ←실제(구사율 5% 미만이라 레퍼토리 밖)

사례 2: Waldichuk, Ken (2023년) -- 0B0S, 아웃 0, 주자 1명, 득점차 -1, 상대 동타


  실제 결과(woba_value): 2.000, gap(실제 예측값 - 최선 예측값): 0.3029
pitch_type  season_usage_rate  predicted_run_value                    실제선택
        ST             0.2707               0.3936                        
        CH             0.1657               0.3938                        
        FF             0.5429               0.4248                        
        CU             0.0204               0.6965 ←실제(구사율 5% 미만이라 레퍼토리 밖)

사례 3: Kikuchi, Yusei (2025년) -- 2B0S, 아웃 0, 주자 1명, 득점차 +2, 상대 반대타
  실제 결과(woba_value): 0.900, gap(실제 예측값 - 최선 예측값): 0.2814
pitch_type  season_usage_rate  predicted_run_value 실제선택
        CH             0.1215               0.4178     
        FF             0.3494               0.4395     
        SL             0.3623               0.4482     
        CU             0.1538               0.6992  ←실제

사례 4: Crawford, Kutter (2022년) -- 0B1S, 아웃 1, 주자 0명, 득점차 -2, 상대 동타
  실제 결과(woba_value): 0.000, gap(실제 예측값 - 최선 예측값): 0.2637
pitch_type  season_usage_rate  predicte

## 10. 투수별 사례

포함 기준(재대결 10건 이상)을 만족한 투수 중 회피율 상위 5명·하위 5명입니다.

In [26]:
name_lookup = xbh.drop_duplicates('pitcher').set_index('pitcher')['pitcher_name']
cases = rates.copy()
cases['pitcher_name'] = cases['pitcher'].map(name_lookup)
cols = ['pitcher_name', 'n_events', 'avoidance_rate', 'mean_rvae', 'mean_repertoire_gap', 'best_match_rate']
print('회피율 상위 5명 (회피형)')
display(cases.sort_values('avoidance_rate', ascending=False)[cols].head(5).round(4))
print('회피율 하위 5명 (승부형)')
display(cases.sort_values('avoidance_rate')[cols].head(5).round(4))

회피율 상위 5명 (회피형)


,pitcher_name,n_events,avoidance_rate,mean_rvae,mean_repertoire_gap,best_match_rate
429,"Schultz, Noah",13,0.9231,0.2070,0.0181,0.2308
372,"Arihara, Kohei",11,0.9091,0.3496,0.0149,0.2727
349,"Schwellenbach, Spencer",42,0.8333,0.0798,0.0174,0.2619
133,"Crismatt, Nabil",17,0.8235,-0.0073,0.0235,0.5294
394,"Brieske, Beau",16,0.8125,-0.0688,0.0110,0.3125


회피율 하위 5명 (승부형)


,pitcher_name,n_events,avoidance_rate,mean_rvae,mean_repertoire_gap,best_match_rate
427,"Yesavage, Trey",10,0.1000,-0.1092,0.0168,0.4000
161,"Widener, Taylor",14,0.1429,0.1123,0.0101,0.4286
213,"McKenzie, Triston",75,0.1467,0.0726,0.0085,0.4533
314,"Abbott, Cory",12,0.1667,0.1936,0.0074,0.5000
436,"Sasaki, Roki",33,0.1818,-0.0334,0.0098,0.3333


In [27]:
print('[레퍼토리 최선 일치율] 3분위:', rates.groupby('tercile_group')['best_match_rate'].mean().round(4).to_dict())

[레퍼토리 최선 일치율] 3분위: {'승부형': 0.38, '중립': 0.3113, '회피형': 0.3099}


## 11. 해석

* **모델 성능.** 노트북 3과 같은 모델(상황+구종+구사율, 실행 품질·`pitcher_id` 제외)입니다 — 검증 세트(2026)
  MAE 0.3989 vs 베이스라인 0.4322, R² = 0.057. 레퍼토리 비교가 쓰는 "기대 Run Value"의 기준선이 이 모델이라,
  노트북 3의 모델 성능 한계(설명력 5.7%, 대부분은 실행 품질·운)가 여기도 그대로 적용됩니다.
* **메인 결과: 전체 일치율.** 재대결 결정구 27,256건(레퍼토리가 1개뿐이라 비교 불가한 135건 제외) 중, 레퍼토리
  평균 크기 4.44개 기준 무작위로 찍어도 맞을 기대 일치율은 23.7%입니다. 실제 일치율(투수별 평균의 평균)은
  **33.4%**로 무작위보다 뚜렷이 높습니다 — "뭘 던졌어야 했는가"라는 기준으로 보면, 투수들의 실제 선택은
  완전히 무작위는 아니고 모델이 보기에 어느 정도 합리적인 쪽으로 쏠려 있습니다. 다만 3건 중 2건은 모델의
  "최선"과는 다른 구종을 택했다는 뜻이기도 합니다. gap(실제 선택의 예측값 − 레퍼토리 최선의 예측값) 분포는
  평균 0.0157, 중앙값 0.0098로 오른쪽으로 길게 늘어져 있습니다(최댓값 0.312) — 대부분은 최선과 큰 차이가
  없지만, 일부는 크게 벌어집니다.
* **회피형 vs 승부형 (보조적 교차 비교).** 3분위로 나누면 승부형 38.0% > 중립 31.1% > 회피형 31.0%로, **개별
  결정 수준에서는 승부형이 더 "모델의 최선"에 가깝게 던집니다** — 노트북 3의 메인 결과(장기 RVAE는 회피형이
  방향상 유리)와 반대 방향입니다. 그럴듯한 설명: 승부형(회피율이 낮은) 투수는 주력 구종에 크게 의존하고, 그
  주력 구종이 구사율 피처 덕분에 모델상으로도 자주 "최선"으로 나오기 쉽습니다. 이 둘이 반대 방향이라는 것
  자체가, "개별 타석에서 레퍼토리 중 최선을 골랐는가"와 "장기적으로 좋은 성향인가"가 **서로 다른 질문이고
  같이 가지 않을 수 있다**는 걸 보여줍니다 — 이게 이 노트북을 노트북 3과 분리해서 보는 이유입니다.
* **구체적 사례 (9절).** gap이 가장 큰 5건 중 4건은 공통된 패턴을 보입니다: 그 시즌 구사율 5% 미만인
  희귀 구종(커브 3건, 스플리터 1건)을 재대결에서 꺼냈다가 장타를 또 맞은 경우입니다(예측 Run Value가 주력
  구종보다 0.2~0.4 가까이 높게 나옴). 즉 모델 기준으로 "가장 아쉬웠던" 선택들은 회피 자체가 아니라
  **잘 안 던지는 구종으로의 회피**였습니다 — 구종을 바꾸는 것 자체보다, 어떤 구종으로 바꾸는지가 더
  중요하다는 뜻으로 읽힙니다. 나머지 1건(Kikuchi)은 반대로 구사율 15%의 주력급 구종이었는데도 그 상황에서는
  모델상 분명히 나쁜 선택이었던 경우입니다.
* **노트북 3과의 관계.** 이 노트북은 노트북 3의 메인 결론("회피 성향이 장기적으로 좋은가")에 대해 아무것도
  바꾸지 않습니다 — 서로 다른 질문에 서로 다른 답을 주는 두 분석입니다. 회의록이 "개별 타석 최적 구종"을
  메인 결론으로 쓰지 말라고 한 이유가 여기서 드러납니다: 그 기준(일치율)으로 보면 승부형이 더 나아 보이고,
  장기 결과(RVAE) 기준으로 보면 회피형이 (유의하진 않지만) 더 나아 보입니다. 하나의 지표만 보고 "이렇게
  던져야 한다"고 결론 내리면 둘 중 하나는 반대로 오도하게 됩니다.
* **한계.** 레퍼토리 기준(구사율 5%)은 사람이 정한 경계값이고, 그 경계 바로 아래의 구종(위 사례들의 커브처럼)이
  실제로는 가장 중요한 변수일 수 있는데 "비교 대상"에서는 빠집니다. gap은 결정구 하나의 상관적 지표라 "그
  구종 선택 하나만의" 순수 인과효과가 아니며, `pitcher_id`를 뺐기 때문에 투수별 실제 구위(구속·무브먼트·
  커맨드)는 구분하지 못하고 `season_usage_rate`가 불완전한 대리 변수입니다. `final_model`은 비교 대상
  시즌을 포함한 전체 데이터로 학습해, 그 시즌에 한해서는 표본 내(in-sample) 예측이라는 점도 감안해야
  합니다. 그리고 "모델의 최선"은 모델이 볼 수 있는 피처(상황+구사율) 안에서의 최선일 뿐, 그 타석 직전까지의
  흐름이나 다음 타자·이닝 운영 같은 더 넓은 맥락은 반영하지 못합니다.